In [4]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

from openai import OpenAI

ollama_url = "http://localhost:11434/v1"

ollama = OpenAI(
    api_key="ollama",
    base_url=ollama_url
)

In [1]:
gpt_model = "gpt-oss:20b"
gemini_model = "gemma3:12b"
llama_model = "llama3.3:70b"


gpt_system = """
You are a highly argumentative and confrontational chatbot.

You disagree with claims made by the other participants whenever you
can find a reasonable basis to challenge them. Question assumptions,
point out contradictions, and push back on weak arguments.

Be snarky, witty, and provocative, but remain intellectually honest.
Do not agree merely to be polite.
"""

gemini_system = """
You are the peacemaker in a heated conversation.

Your goal is to reduce conflict and keep the discussion productive.
Look for common ground between the participants and acknowledge
valid points from both sides.

When participants attack each other, calmly reframe their arguments
and encourage constructive discussion.

Be warm, patient, and diplomatic.
"""

llama_system = """
You are the confused mediator in a debate between two other chatbots.

You try to understand what everyone is arguing about, summarize their
positions, and occasionally misunderstand something in an amusing way.

When the argument gets heated, attempt to mediate, but don't always
succeed. Ask pointed questions when the other participants contradict
themselves.

Be friendly, slightly confused, and unintentionally funny.
"""

In [ ]:
!ollama pull gpt-oss:20b
!ollama pull gemma3:12b
!ollama pull llama3.3:70b

In [5]:
def call_model(model, system_prompt, messages):
    
    conversation = [
        {"role": "system", "content": system_prompt}
    ] + messages

    response = ollama.chat.completions.create(
        model=model,
        messages=conversation
    )

    return response.choices[0].message.content

In [6]:
topic = "Should AI replace software engineers?"

In [7]:
debate_messages = [
    {"role": "user", "content": topic}
]

In [ ]:
for round_number in range(3):

    print(f"\n========== ROUND {round_number + 1} ==========\n")

    gpt_response = call_model(
        gpt_model,
        gpt_system,
        debate_messages
    )

    display(Markdown(f"### 😈 GPT\n{gpt_response}"))


    gemini_messages = debate_messages + [
        {"role": "user", "content": f"GPT says:\n{gpt_response}"}
    ]

    gemma_response = call_model(
        gemini_model,
        gemini_system,
        gemini_messages
    )

    display(Markdown(f"### 🕊️ Gemma\n{gemma_response}"))


    llama_messages = debate_messages + [
        {"role": "user", "content": f"GPT says:\n{gpt_response}"},
        {"role": "user", "content": f"Gemma says:\n{gemma_response}"}
    ]

    llama_response = call_model(
        llama_model,
        llama_system,
        llama_messages
    )

    display(Markdown(f"### 🤝 Llama\n{llama_response}"))

    debate_messages.extend([
        {"role": "user", "content": f"GPT says:\n{gpt_response}"},
        {"role": "user", "content": f"Gemma says:\n{gemma_response}"},
        {"role": "user", "content": f"Llama says:\n{llama_response}"}
    ])